# Phase 9: Confidence Service Checks

Validates the confidence scoring pipeline:
- Calibrated confidence scores for 10 benchmark claims
- Per-source edge confidence weights
- Calibration curve (raw vs calibrated)
- Verification that correlated context does NOT inflate scores

Uses Groq API (instant inference). Falls back to local Qwen if GROQ_API_KEY is not set.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.app.models.calibration_model import calibrate
from backend.app.models.llm_model import get_groq_llm_model, get_llm_model, LLMResult
from backend.app.models.nli_model import NLIModel, NLIResult
from backend.app.services.cache_service import CacheService
from backend.app.services.confidence_service import ConfidenceService, ConfidenceOutput
from backend.app.services.evidence_expansion_service import EvidenceExpansionService
from backend.app.services.ranking_service import RankingService
from backend.app.services.retrieval_service import RetrievalService
from backend.app.services.stance_service import StanceService
from backend.app.retrieval.retriever_registry import RetrieverRegistry
from backend.app.retrieval.wikipedia_retriever import WikipediaRetriever
from backend.app.retrieval.factcheck_retriever import FactCheckRetriever
from backend.app.retrieval.guardian_retriever import GuardianRetriever
from backend.app.retrieval.newsapi_retriever import NewsApiRetriever
from backend.app.retrieval.gdelt_retriever import GDELTRetriever
from backend.app.retrieval.livewiki_retriever import LiveWikiRetriever
from backend.app.utils.constants import (
    DEFAULT_RETRIEVAL_CACHE_DIR,
    GROQ_MODEL_NAME,
    LLM_MAX_INPUT_SOURCES,
    NLI_MODEL_NAME,
    NLI_CONFIRM_MODEL_NAME,
    CALIBRATION_BREAKPOINTS,
    FEVER_EVIDENCE_SNIPPETS_OUTPUT_FILENAME,
    DEFAULT_PROCESSED_DIR,
)

import os
from dotenv import load_dotenv
load_dotenv()

print("Imports OK")

/Users/vraj21/Desktop/Projects/Fact Checker/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Imports OK


## Load Services

In [2]:
# Build retrieval service
registry = RetrieverRegistry()
evidence_path = DEFAULT_PROCESSED_DIR / FEVER_EVIDENCE_SNIPPETS_OUTPUT_FILENAME
registry.register(WikipediaRetriever(evidence_snippets_path=evidence_path))
registry.register(FactCheckRetriever(api_key=os.getenv('FACTCHECK_API_KEY')))
registry.register(GuardianRetriever(api_key=os.getenv('GUARDIAN_API_KEY')))
registry.register(NewsApiRetriever(api_key=os.getenv('NEWSAPI_KEY')))
registry.register(GDELTRetriever())
registry.register(LiveWikiRetriever())

cache = CacheService(DEFAULT_RETRIEVAL_CACHE_DIR)
retrieval_svc = RetrievalService(
    registry=registry, cache=cache,
    ranking=RankingService(), expansion=EvidenceExpansionService()
)

# NLI
nli_model = NLIModel(model_name=NLI_MODEL_NAME)
stance_svc = StanceService(model=nli_model, cache=cache)

# LLM (Groq preferred)
try:
    llm = get_groq_llm_model(model_name=GROQ_MODEL_NAME)
    print(f"LLM: Groq ({llm.model_name})")
except ValueError:
    llm = get_llm_model()
    print(f"LLM: Local ({llm.model_name})")

# Confidence
confidence_svc = ConfidenceService()
print("All services loaded.")

Loading weights: 100%|██████████| 106/106 [00:00<00:00, 1808.28it/s, Materializing param=pooler.dense.weight]                                     
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Groq] Model: llama-3.1-8b-instant
LLM: Groq (llama-3.1-8b-instant)
All services loaded.


## 10-Claim Benchmark

In [3]:
BENCHMARK = [
    ("The Eiffel Tower is located in Paris, France.", "verified"),
    ("Water boils at 100 degrees Celsius at sea level.", "verified"),
    ("Barack Obama was the 44th president of the United States.", "verified"),
    ("The Great Wall of China is visible from space with the naked eye.", "rejected"),
    ("Humans only use 10 percent of their brains.", "rejected"),
    ("A secret underground city exists beneath Denver Airport.", "not_enough_info"),
    ("The number of cats in Tokyo exceeded 5 million in 2025.", "not_enough_info"),
    ("Coffee is good for your health.", "not_enough_info"),
    ("Einstein failed math in school.", "rejected"),
    ("Henri Christophe built a palace in Milot.", "verified"),
]

print(f"{len(BENCHMARK)} claims loaded.")

10 claims loaded.


In [4]:
results = []

for claim_text, expected in BENCHMARK:
    print(f"\n{'='*70}")
    print(f"CLAIM: {claim_text}")
    print(f"{'='*70}")
    print(f"  Expected: {expected}")

    sources = retrieval_svc.retrieve(claim_text, max_results=10, use_cache=False)
    if not sources:
        print("  [WARN] No sources.")
        results.append({"claim": claim_text, "expected": expected, "verdict": "skip", "conf": 0.0, "passed": False})
        continue

    classified, nli_results = stance_svc.classify(claim_text, sources)
    llm_input = classified[:LLM_MAX_INPUT_SOURCES]
    llm_result = llm.classify(claim_text, llm_input)

    conf = confidence_svc.compute_main_confidence(llm_result, llm_input, nli_results)

    print(f"  LLM verdict  : {llm_result.overall_verdict} (conf={llm_result.confidence:.2f})")
    print(f"  SUB-SCORES:")
    print(f"    directional : {conf.debug['directional']:.2f} (support={conf.support_score:.2f}, refute={conf.refute_score:.2f})")
    print(f"    llm_conf    : {conf.debug['llm_conf']:.2f}")
    print(f"    quality     : {conf.evidence_quality:.2f}")
    print(f"    corroborate : {conf.corroboration:.2f} ({len(conf.debug.get('agreeing_types', []))} types)")
    print(f"    coverage    : {conf.coverage:.2f}")
    print(f"  RAW -> CAL    : {conf.raw_confidence:.2f} -> {conf.overall_confidence:.2f}")
    print(f"  FINAL         : {conf.overall_verdict} ({conf.overall_confidence:.2f})")

    # Edge confidences
    class_by_idx = {sc.index: sc for sc in llm_result.sources}
    edge_confs = []
    for i, src in enumerate(llm_input, start=1):
        sc = class_by_idx.get(i)
        llm_class = sc.classification if sc else "insufficient"
        nli = nli_results.get(i - 1)
        edge = confidence_svc.compute_edge_confidence(src, llm_class, nli)
        edge_confs.append((llm_class, edge, src.source_type))
        marker = " <-- lower" if llm_class == "correlated_context" else ""
        print(f"    [{i}] {llm_class:<22} edge={edge:.2f}  {src.source_type}{marker}")

    passed = conf.overall_verdict == expected
    label = "PASS" if passed else "FAIL"
    print(f"  [{label}]")

    results.append({
        "claim": claim_text, "expected": expected,
        "verdict": conf.overall_verdict, "conf": conf.overall_confidence,
        "passed": passed, "conf_output": conf, "edges": edge_confs,
    })

Retriever 'wikipedia' failed for query='The Eiffel Tower is located in Paris, France.': evidence snippets parquet not found: data/processed/fever_evidence_snippets.parquet



CLAIM: The Eiffel Tower is located in Paris, France.
  Expected: verified


Retriever 'newsapi' failed for query='The Eiffel Tower is located in Paris, France.': 429 Client Error: Too Many Requests for url: https://newsapi.org/v2/everything?q=The+Eiffel+Tower+is+located+in+Paris%2C+France.&language=en&sortBy=relevancy&pageSize=25&page=1&searchIn=title%2Cdescription
Partial retrieval: 2 retrievers failed (['wikipedia', 'newsapi']). Proceeding with 17 raw results from 4 sources.


[Groq] Generating... (model=llama-3.1-8b-instant, max_tokens=512)


Retriever 'wikipedia' failed for query='Water boils at 100 degrees Celsius at sea level.': evidence snippets parquet not found: data/processed/fever_evidence_snippets.parquet


...

[Groq] Generating... (model=llama-3.1-8b-instant, max_tokens=512)


Retriever 'wikipedia' failed for query='Henri Christophe built a palace in Milot.': evidence snippets parquet not found: data/processed/fever_evidence_snippets.parquet


[Groq] Generation complete.
  LLM verdict  : insufficient (conf=0.00)
  SUB-SCORES:
    directional : 0.00 (support=0.00, refute=0.00)
    llm_conf    : 0.00
    quality     : 0.00
    corroborate : 0.00 (0 types)
    coverage    : 0.25
  RAW -> CAL    : 0.01 -> 0.05
  FINAL         : not_enough_info (0.05)
    [1] insufficient           edge=0.83  guardian
    [2] insufficient           edge=0.71  guardian
  [FAIL]

CLAIM: Henri Christophe built a palace in Milot.
  Expected: verified


Retriever 'newsapi' failed for query='Henri Christophe built a palace in Milot.': 429 Client Error: Too Many Requests for url: https://newsapi.org/v2/everything?q=Henri+Christophe+built+a+palace+in+Milot.&language=en&sortBy=relevancy&pageSize=25&page=1&searchIn=title%2Cdescription
Partial retrieval: 2 retrievers failed (['wikipedia', 'newsapi']). Proceeding with 15 raw results from 4 sources.


[Groq] Generating... (model=llama-3.1-8b-instant, max_tokens=512)
[Groq] Generation complete.
  LLM verdict  : supported (conf=1.00)
  SUB-SCORES:
    directional : 0.99 (support=0.99, refute=0.00)
    llm_conf    : 1.00
    quality     : 0.48
    corroborate : 0.00 (1 types)
    coverage    : 0.25
  RAW -> CAL    : 0.71 -> 0.61
  FINAL         : verified (0.61)
    [1] direct_support         edge=0.85  livewiki
    [2] correlated_context     edge=0.69  livewiki <-- lower
  [PASS]


## Summary Table

In [5]:
print(f"{'#':<3} {'Expected':<16} {'Actual':<16} {'Conf':<8} {'Pass':<6} Claim")
print("-" * 80)
total_pass = 0
for i, r in enumerate(results, 1):
    if r['passed']:
        total_pass += 1
    status = 'PASS' if r['passed'] else ('SKIP' if r['verdict'] == 'skip' else 'FAIL')
    print(f"{i:<3} {r['expected']:<16} {r['verdict']:<16} {r['conf']:<8.2f} {status:<6} {r['claim'][:50]}")

print(f"\nTotal: {total_pass}/{len(results)} passed")

#   Expected         Actual           Conf     Pass   Claim
--------------------------------------------------------------------------------
1   verified         verified         0.79     PASS   The Eiffel Tower is located in Paris, France.
2   verified         not_enough_info  0.05     FAIL   Water boils at 100 degrees Celsius at sea level.
3   verified         verified         0.71     PASS   Barack Obama was the 44th president of the United 
4   rejected         not_enough_info  0.10     FAIL   The Great Wall of China is visible from space with
5   rejected         rejected         0.50     PASS   Humans only use 10 percent of their brains.
6   not_enough_info  not_enough_info  0.05     PASS   A secret underground city exists beneath Denver Ai
7   not_enough_info  not_enough_info  0.05     PASS   The number of cats in Tokyo exceeded 5 million in 
8   not_enough_info  not_enough_info  0.06     PASS   Coffee is good for your health.
9   rejected         not_enough_info  0.05     FAIL 

## Calibration Curve

In [6]:
import numpy as np

# Plot calibration function
raw_vals = np.linspace(0, 1, 200)
cal_vals = [calibrate(x) for x in raw_vals]

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(raw_vals, cal_vals, 'b-', linewidth=2, label='Calibration curve')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Identity (no calibration)')

    # Plot breakpoints
    bx = [b[0] for b in CALIBRATION_BREAKPOINTS]
    by = [b[1] for b in CALIBRATION_BREAKPOINTS]
    ax.scatter(bx, by, c='red', zorder=5, label='Breakpoints')

    # Plot actual results
    for r in results:
        if 'conf_output' in r:
            co = r['conf_output']
            color = 'green' if r['passed'] else 'red'
            ax.scatter(co.raw_confidence, co.overall_confidence, c=color, s=80, zorder=6, edgecolors='black')

    ax.set_xlabel('Raw Confidence')
    ax.set_ylabel('Calibrated Confidence')
    ax.set_title('Confidence Calibration Curve')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skipping plot.")
    print("\nCalibration values (sampled):")
    for x in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        print(f"  raw={x:.1f} -> cal={calibrate(x):.3f}")

matplotlib not installed — skipping plot.

Calibration values (sampled):
  raw=0.0 -> cal=0.050
  raw=0.1 -> cal=0.083
  raw=0.2 -> cal=0.117
  raw=0.3 -> cal=0.150
  raw=0.4 -> cal=0.275
  raw=0.5 -> cal=0.400
  raw=0.6 -> cal=0.500
  raw=0.7 -> cal=0.600
  raw=0.8 -> cal=0.733
  raw=0.9 -> cal=0.850
  raw=1.0 -> cal=0.950


## Automated Checks

In [7]:
all_pass = True

# Check 1: No confidence is exactly 0.0 or 1.0
for r in results:
    if r['verdict'] == 'skip':
        continue
    if r['conf'] == 0.0 or r['conf'] == 1.0:
        print(f"[FAIL] Confidence is exactly {r['conf']} for: {r['claim'][:50]}")
        all_pass = False

# Check 2: Correlated sources have lower edge confidence than direct sources (per claim)
for r in results:
    if 'edges' not in r:
        continue
    direct_edges = [e[1] for e in r['edges'] if e[0] in ('direct_support', 'direct_refute')]
    correlated_edges = [e[1] for e in r['edges'] if e[0] == 'correlated_context']
    if direct_edges and correlated_edges:
        if max(correlated_edges) >= min(direct_edges):
            print(f"[WARN] Correlated edge >= direct edge for: {r['claim'][:50]}")

# Check 3: Calibration floor/ceiling
assert calibrate(0.0) > 0.0, "Calibration floor broken"
assert calibrate(1.0) < 1.0, "Calibration ceiling broken"
print(f"Calibration floor: {calibrate(0.0):.2f}, ceiling: {calibrate(1.0):.2f}")

if all_pass:
    print("\n[PASS] All automated checks passed.")
else:
    print("\n[WARN] Some checks failed — review above.")

Calibration floor: 0.05, ceiling: 0.95

[PASS] All automated checks passed.


## Final Checklist

- [ ] `calibrate()` enforces floor (0.05) and ceiling (0.95)
- [ ] Verified claims produce confidence >= 0.50
- [ ] NEI claims produce confidence <= 0.45
- [ ] Correlated context sources have lower edge confidence than direct sources
- [ ] No claim produces confidence exactly 0.0 or 1.0
- [ ] `--mode confidence` CLI runs end-to-end without errors
- [ ] Calibration curve shows S-shaped compression at extremes
- [ ] LLM confidence floor prevents 0.0 confidence for claims with direct evidence

**Note**: Claims failing due to weak retrieval (no relevant snippets found) are
expected to produce NEI — this is honest behavior, not a confidence service bug.
The confidence service cannot create evidence that doesn't exist.